# AIE223 Poker Card Detector — Kaggle Version

> **หมายเหตุ:** Notebook นี้ดัดแปลงจาก Colab version เพื่อรันบน Kaggle
> ความแตกต่างหลัก: path เปลี่ยนเป็น `/kaggle/working/`, ลบ `google.colab` ออก, ใช้ Kaggle Secrets สำหรับ API key

### ลำดับการรัน
1. Cell 1 — ติดตั้ง dependencies
2. Cell 2 — Download dataset จาก Roboflow
3. Cell 3 — ตรวจสอบโครงสร้าง dataset
4. Cell 4 — Rename class `3_of_hearts` → `3H`
5. Cell 5 — Resplit dataset: Train 70% / Val 20% / Test 10%
6. Cell 6 — Train model ด้วย YOLO26n
7. Cell 7 — Evaluate บน Val และ Test set
8. Cell 8 — แสดงผลกราฟ
9. Cell 9 — รวมไฟล์ output ไว้ใน `/kaggle/working/` (download ผ่าน Output panel)

### วิธีตั้งค่า Roboflow API Key บน Kaggle
1. ไปที่ **Add-ons → Secrets** ใน Kaggle Notebook
2. กด **+ Add a new secret**
3. Name: `ROBOFLOW_API_KEY`, Value: ใส่ key ของคุณ
4. เปิด toggle **Attach to notebook**

## ติดตั้ง dependencies

In [ ]:
!pip install -U ultralytics roboflow -q

import ultralytics
print(f'Ultralytics version: {ultralytics.__version__}')

import torch
DEVICE = '0,1' if torch.cuda.is_available() else 'cpu'
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB')
else:
    print('Running on CPU (ช้ากว่า GPU ~10x)')

# กำหนด base working dir สำหรับ Kaggle
WORKDIR = '/kaggle/working'
print(f'Working dir: {WORKDIR}')
print('Dependencies ready ✅')

## Download dataset จาก Roboflow

In [ ]:
# ── วิธีที่ 1 (แนะนำ): ใช้ Kaggle Secrets ──────────────────────────────────
# ไปที่ Add-ons → Secrets แล้วเพิ่ม key ชื่อ ROBOFLOW_API_KEY ก่อนรัน
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    ROBOFLOW_API_KEY = user_secrets.get_secret('ROBOFLOW_API_KEY')
    print('✅ โหลด API key จาก Kaggle Secrets สำเร็จ')
except Exception:
    # ── วิธีที่ 2 (fallback): ใส่ key ตรงๆ ─────────────────────────────────
    ROBOFLOW_API_KEY = 'inImw89oDETWFt75zUEh'  # ← เปลี่ยนเป็น key ของคุณ
    print('⚠️  ใช้ API key แบบ hardcode (แนะนำให้ย้ายไป Kaggle Secrets)')

import os
from roboflow import Roboflow

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.project('poker-gq2zv')
dataset = project.version(1).download('yolov8')

ORIG = dataset.location
print(f'Dataset downloaded: {ORIG}')

## ตรวจสอบโครงสร้าง dataset

In [ ]:
import os, yaml

DATA_YAML = os.path.join(ORIG, 'data.yaml')

print('Dataset structure:')
for item in sorted(os.listdir(ORIG)):
    print(f'  {item}')

with open(DATA_YAML, 'r') as f:
    data = yaml.safe_load(f)

print(f'\nnc: {data["nc"]}')
print(f'\nClass names:')
for i, name in enumerate(data['names']):
    flag = ' ← ต้องแก้!' if name == '3_of_hearts' else ''
    print(f'  [{i:2d}] {name}{flag}')

print('\nImage counts per split:')
total = 0
for split in ['train', 'valid', 'test']:
    img_dir = os.path.join(ORIG, split, 'images')
    count = len(os.listdir(img_dir)) if os.path.exists(img_dir) else 0
    total += count
    print(f'  {split:6s}: {count} images')
print(f'  Total : {total} images')

## Rename class '3_of_hearts' → '3H'

In [ ]:
import yaml

DATA_YAML = os.path.join(ORIG, 'data.yaml')

with open(DATA_YAML, 'r') as f:
    data = yaml.safe_load(f)

if '3_of_hearts' in data['names']:
    idx = data['names'].index('3_of_hearts')
    data['names'][idx] = '3H'
    with open(DATA_YAML, 'w') as f:
        yaml.dump(data, f, sort_keys=False, allow_unicode=True)
    print(f"Renamed: '3_of_hearts' → '3H' at index {idx}")
else:
    print("'3_of_hearts' ไม่พบ — ชื่อถูกต้องอยู่แล้ว")

print(f'\nindex 11 ตอนนี้: {data["names"][11]}')
print(f'Class names ทั้งหมด: {data["names"]}')

## Resplit dataset → Train 70% / Val 20% / Test 10%
> รวมทุก split เดิม (train+valid+test) แล้วแบ่งใหม่
> เพื่อแก้ปัญหา val set เล็กเกินไปและได้ test set แยกจริงๆ

In [ ]:
import os, shutil, random, yaml

# ── เปลี่ยนจาก /content/ → /kaggle/working/ ──────────────────────────────
NEW = '/kaggle/working/Poker-final'

if os.path.exists(NEW):
    shutil.rmtree(NEW)

for split in ['train', 'valid', 'test']:
    os.makedirs(f'{NEW}/{split}/images', exist_ok=True)
    os.makedirs(f'{NEW}/{split}/labels', exist_ok=True)

all_files = []
for split in ['train', 'valid', 'test']:
    img_dir = f'{ORIG}/{split}/images'
    if os.path.exists(img_dir):
        for fname in os.listdir(img_dir):
            if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                all_files.append((split, fname))

print(f'Total images collected: {len(all_files)}')

random.seed(42)
random.shuffle(all_files)

n = len(all_files)
n_train = int(n * 0.70)
n_val   = int(n * 0.20)

train_files = all_files[:n_train]
val_files   = all_files[n_train:n_train + n_val]
test_files  = all_files[n_train + n_val:]

print(f'Train : {len(train_files)} images (70%)')
print(f'Val   : {len(val_files)} images (20%)')
print(f'Test  : {len(test_files)} images (10%)')

def copy_split(file_list, dest_split):
    copied = 0
    for src_split, fname in file_list:
        for sub, fn in [('images', fname),
                        ('labels', os.path.splitext(fname)[0] + '.txt')]:
            src = f'{ORIG}/{src_split}/{sub}/{fn}'
            dst = f'{NEW}/{dest_split}/{sub}/{fn}'
            if os.path.exists(src):
                shutil.copy2(src, dst)
                if sub == 'images':
                    copied += 1
    return copied

copy_split(train_files, 'train')
copy_split(val_files,   'valid')
copy_split(test_files,  'test')

with open(f'{ORIG}/data.yaml', 'r') as f:
    orig_data = yaml.safe_load(f)

# ── path ใน data.yaml ต้องใช้ /kaggle/working/ ───────────────────────────
new_data = {
    'train' : f'{NEW}/train/images',
    'val'   : f'{NEW}/valid/images',
    'test'  : f'{NEW}/test/images',
    'nc'    : 52,
    'names' : orig_data['names']
}

NEW_YAML = f'{NEW}/data.yaml'
with open(NEW_YAML, 'w') as f:
    yaml.dump(new_data, f, sort_keys=False, allow_unicode=True)

print(f'\ndata.yaml saved: {NEW_YAML}')
print('Dataset ready ✅')

## Train YOLO26n

In [ ]:
from ultralytics import YOLO
import torch

# ── เปลี่ยน path จาก /content/ → /kaggle/working/ ────────────────────────
NEW_YAML = '/kaggle/working/Poker-final/data.yaml'
DEVICE   = 0 if torch.cuda.is_available() else 'cpu'

print('Loading YOLO26n pretrained weights...')
model = YOLO('yolo26n.pt')

print(f'Starting training on device: {DEVICE}\n')

results = model.train(
    data    = NEW_YAML,
    epochs  = 150,
    imgsz   = 640,
    batch   = 16,
    device  = DEVICE,
    patience= 30,
    save    = True,
    project = '/kaggle/working/poker_training',   # ← เปลี่ยน path
    name    = 'run_yolo26',
    verbose = True,
    plots   = True,
    workers = 4     # ← Kaggle มี 4 CPU cores (Colab มี 2)
)

print('\n' + '=' * 60)
print('TRAINING COMPLETE')
print('=' * 60)
print(f'Best mAP@50  : {results.results_dict["metrics/mAP50(B)"]:.1%}')
print(f'Precision    : {results.results_dict["metrics/precision(B)"]:.1%}')
print(f'Recall       : {results.results_dict["metrics/recall(B)"]:.1%}')
print(f'Weights saved: {results.save_dir}/weights/best.pt')
print('=' * 60)

## Evaluate บน Val และ Test set

In [ ]:
from ultralytics import YOLO
import torch, json, os
from pathlib import Path

WEIGHT_PATH = str(results.save_dir) + '/weights/best.pt'
# ── เปลี่ยน path จาก /content/ → /kaggle/working/ ────────────────────────
DATA_YAML   = '/kaggle/working/Poker-final/data.yaml'
DEVICE      = 0 if torch.cuda.is_available() else 'cpu'

print(f'Evaluating: {WEIGHT_PATH}\n')
eval_model = YOLO(WEIGHT_PATH)
evaluation_results = {}

for split in ['val', 'test']:
    try:
        metrics = eval_model.val(
            data      = DATA_YAML,
            split     = split,
            imgsz     = 640,
            batch     = 16,
            device    = DEVICE,
            plots     = True,
            save_json = True,
            project   = '/kaggle/working/poker_evaluation',  # ← เปลี่ยน path
            name      = f'eval_{split}'
        )

        evaluation_results[split] = {
            'model'            : 'YOLO26n',
            'precision'        : round(float(metrics.box.mp)    * 100, 2),
            'recall'           : round(float(metrics.box.mr)    * 100, 2),
            'mAP50'            : round(float(metrics.box.map50) * 100, 2),
            'mAP50_95'         : round(float(metrics.box.map)   * 100, 2),
            'speed_inference_ms': round(float(metrics.speed.get('inference', 0.0)), 2),
            'save_dir'         : str(metrics.save_dir),
        }

        r = evaluation_results[split]
        print('\n' + '=' * 55)
        print(f' EVALUATION — {split.upper()} SPLIT (YOLO26n)')
        print('=' * 55)
        print(f"  Precision : {r['precision']:.2f}%")
        print(f"  Recall    : {r['recall']:.2f}%")
        print(f"  mAP@50    : {r['mAP50']:.2f}%")
        print(f"  mAP@50-95 : {r['mAP50_95']:.2f}%")
        print(f"  Inference : {r['speed_inference_ms']:.2f} ms/image")
        print(f"  Plots saved: {r['save_dir']}")

    except Exception as exc:
        evaluation_results[split] = {'error': str(exc)}
        print(f'ข้าม {split}: {exc}')

# บันทึก JSON ไว้ใน /kaggle/working/ (จะโชว์ใน Output panel อัตโนมัติ)
json_path = '/kaggle/working/evaluation_yolo26.json'
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(evaluation_results, f, ensure_ascii=False, indent=2)

print(f'\nSaved: {json_path}')
evaluation_results

## แสดงผลกราฟ Training + Evaluation

In [ ]:
from IPython.display import Image, display
import os

train_dir = str(results.save_dir)

print('=== Training Graphs ===')
for fname in ['results.png', 'confusion_matrix.png',
              'PR_curve.png', 'F1_curve.png', 'labels.jpg']:
    path = os.path.join(train_dir, fname)
    if os.path.exists(path):
        print(f'\n--- {fname} ---')
        display(Image(path))

print('\n=== Evaluation Graphs ===')
for split in ['val', 'test']:
    # ── เปลี่ยน path จาก /content/ → /kaggle/working/ ──────────────────
    base = f'/kaggle/working/poker_evaluation/eval_{split}'
    if os.path.exists(base):
        for fname in ['confusion_matrix.png', 'PR_curve.png', 'F1_curve.png']:
            path = os.path.join(base, fname)
            if os.path.exists(path):
                print(f'\n--- {split.upper()}: {fname} ---')
                display(Image(path))

## รวมไฟล์ output (แทน google.colab download)
> บน Kaggle ไม่มี `google.colab` — ใช้วิธี zip แล้ว save ไว้ใน `/kaggle/working/`
> จากนั้น download ได้เลยผ่าน **Output panel** ทางขวา

In [ ]:
# ── ไม่ใช้ google.colab.files อีกต่อไป ───────────────────────────────────
# บน Kaggle: ทุกไฟล์ใน /kaggle/working/ โชว์ใน Output panel อัตโนมัติ
import os, shutil, zipfile

OUTPUT_ZIP = '/kaggle/working/poker_results.zip'
train_dir  = str(results.save_dir)

collected = []

# best.pt
best_pt = os.path.join(train_dir, 'weights/best.pt')
if os.path.exists(best_pt):
    collected.append(('best.pt', best_pt))

# training graphs
for fname in ['results.png', 'confusion_matrix.png', 'PR_curve.png', 'F1_curve.png']:
    path = os.path.join(train_dir, fname)
    if os.path.exists(path):
        collected.append((fname, path))

# evaluation JSON
json_path = '/kaggle/working/evaluation_yolo26.json'
if os.path.exists(json_path):
    collected.append(('evaluation_yolo26.json', json_path))

# evaluation plots
for split in ['val', 'test']:
    base = f'/kaggle/working/poker_evaluation/eval_{split}'
    if os.path.exists(base):
        for fname in ['confusion_matrix.png', 'PR_curve.png', 'F1_curve.png']:
            path = os.path.join(base, fname)
            if os.path.exists(path):
                collected.append((f'eval_{split}_{fname}', path))

# สร้าง zip
with zipfile.ZipFile(OUTPUT_ZIP, 'w', zipfile.ZIP_DEFLATED) as zf:
    for arcname, filepath in collected:
        zf.write(filepath, arcname)
        print(f'  + {arcname}  ({os.path.getsize(filepath)/1024:.0f} KB)')

zip_size = os.path.getsize(OUTPUT_ZIP) / 1024 / 1024
print(f'\n✅ สร้าง {OUTPUT_ZIP} ({zip_size:.1f} MB)')
print('📥 Download ได้เลยจาก Output panel ทางขวาของหน้า Kaggle')
print('   หรือ path: /kaggle/working/poker_results.zip')